# Open shop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PyJobShop/PyJobShop/blob/main/examples/open_shop.ipynb)

> If you're using this notebook in Google Colab, be sure to install PyJobShop first by executing ```pip install pyjobshop``` in a cell.

In this notebook, we demonstrate how to model and solve the open shop problem (OSP) using PyJobShop. The OSP is a classic scheduling environment in which a set of jobs has to be processed on a set of machines, but, unlike the flow shop or job shop, there is no fixed routing: the operations of a job may be processed in any order ([Gonzalez & Sahni, 1976](https://dl.acm.org/doi/10.1145/321978.321985)).

## Problem description

The OSP is characterized as follows:

- There is a set of $n$ jobs that need to be processed on $m$ machines.
- Each job consists of $m$ operations, one for each machine, with a given processing time.
- The operations of a job may be processed in any order (there is no routing).
- Each machine processes one operation at a time.
- Each job is processed on one machine at a time, so its operations cannot overlap.
- The objective is to minimize the makespan.

The open shop differs from the flow shop and job shop only in that it imposes *no* precedence between a job's operations. The two remaining requirements are both _disjunctive_ (no-overlap) requirements:

1. each machine processes one operation at a time, and
2. each job is processed on one machine at a time.

In PyJobShop, a machine is a unit-capacity resource that processes one task at a time, so the first requirement is handled simply by assigning each operation to its machine. A `Job` in PyJobShop groups tasks for the objective (for example, to compute tardiness) but does *not* prevent its tasks from overlapping in time. To enforce the second requirement, we therefore model each job as an additional unit-capacity machine that all of the job's operations require. An operation then occupies both its physical machine *and* its job resource, so no two operations of the same job can run at the same time.

## Data

The data for an OSP is given by a processing times matrix, where entry $(j, k)$ is the processing time of job $j$ on machine $k$:

In [ ]:
DURATIONS = [
    [54, 34, 61, 2],
    [9, 15, 89, 70],
    [38, 19, 28, 87],
    [95, 34, 7, 29],
]

num_jobs, num_machines = len(DURATIONS), len(DURATIONS[0])
print(f"Problem size: {num_jobs} jobs, {num_machines} machines")

## Model

We start by creating the model and adding one machine per physical machine:

In [ ]:
from pyjobshop import Model

model = Model()
machines = [
    model.add_machine(name=f"Machine {k}") for k in range(num_machines)
]

Next, we add one unit-capacity machine per job. These are auxiliary resources whose only purpose is to ensure that the operations of a job do not overlap in time:

In [ ]:
job_machines = [model.add_machine(name=f"Job {j}") for j in range(num_jobs)]

For each job and machine, we create an operation (a task) and a single processing mode. The mode requires *both* the physical machine and the job's resource, so the operation occupies one of each. We deliberately add no precedence constraints between a job's operations, which is exactly what makes this an open shop: the operations may be processed in any order.

In [ ]:
tasks = {}  # store for later

for job_idx in range(num_jobs):
    for machine_idx in range(num_machines):
        task = model.add_task(name=f"({job_idx}, {machine_idx})")
        tasks[job_idx, machine_idx] = task

        machine = machines[machine_idx]
        job_machine = job_machines[job_idx]
        duration = DURATIONS[job_idx][machine_idx]
        model.add_mode(task, [machine, job_machine], duration=duration)

The makespan is the default objective, so we can solve the model directly:

In [ ]:
result = model.solve(display=False)
print(result)

Let's plot the solution. PyJobShop plots every resource, so the figure shows both the physical machines and the auxiliary job resources:

In [ ]:
from pyjobshop.plot import plot_machine_gantt

plot_machine_gantt(result.best, model.data())

The top $m$ rows are the physical machines: no two operations overlap on a machine. The bottom $n$ rows are the job resources, which show the same operations grouped by job: no two operations of a job overlap either, but they appear in no particular machine order. This is precisely the open shop: a job's operations are free to be processed in any order, as long as the job is never on two machines at once.

As a sanity check, the makespan can never be smaller than the most loaded machine or the most loaded job, since those operations must be processed sequentially. We verify that our solution respects this lower bound:

In [ ]:
max_machine_load = max(
    sum(DURATIONS[j][k] for j in range(num_jobs)) for k in range(num_machines)
)
max_job_load = max(sum(DURATIONS[j]) for j in range(num_jobs))
lower_bound = max(max_machine_load, max_job_load)

print(f"Most loaded machine: {max_machine_load}")
print(f"Most loaded job:     {max_job_load}")
print(f"Lower bound:         {lower_bound}")
print(f"Makespan:            {result.objective}")
assert result.objective >= lower_bound

## Conclusion

This notebook demonstrated how to model the open shop problem using PyJobShop. The key idea is that the open shop is a purely disjunctive problem: it has no routing, only no-overlap requirements. We modeled the two requirements with two sets of unit-capacity machines, one for the physical machines and one for the jobs, and let each operation require one of each. Because there are no precedence constraints, the solver is free to sequence each job's operations in any order.